# Cloud AI — Colab backend

Runs **Ollama** and the **Cloud AI router** (`backend/server.py`) on this Colab GPU and exposes them through a Cloudflare quick tunnel.

1. *Runtime → Change runtime type → GPU* (L4 or A100 recommended for the default 27B/30B models).
2. *Runtime → Run all*.
3. The last cell prints `CLOUD_URL` and `CLOUD_TOKEN`. In VS Code open *Settings → Cloud AI* and paste them into **Tunnel Url** and **Token**.

Keep this notebook running while you work. The tunnel URL changes every time the runtime restarts.


In [ ]:
#@title 1 · Configuration
CODE_MODEL   = "qwen3:30b"   #@param {type:"string"}
VISION_MODEL = "gemma3:27b"  #@param {type:"string"}
REPO         = "https://github.com/cloud-fronts-group/cloud-ai"  #@param {type:"string"}

import os, secrets, subprocess, time, re, urllib.request
os.environ["CLOUD_CODE_MODEL"] = CODE_MODEL
os.environ["CLOUD_VISION_MODEL"] = VISION_MODEL
os.environ["CLOUD_TOKEN"] = os.environ.get("CLOUD_TOKEN") or secrets.token_urlsafe(24)
os.environ["OLLAMA_HOST"] = "http://127.0.0.1:11434"
!nvidia-smi --query-gpu=name,memory.total --format=csv


In [ ]:
#@title 2 · Install and start Ollama
!curl -fsSL https://ollama.com/install.sh | sh > /tmp/ollama-install.log 2>&1 && echo "ollama installed"
ollama_proc = subprocess.Popen(
    ["ollama", "serve"],
    stdout=open("/tmp/ollama.log", "w"), stderr=subprocess.STDOUT,
    env={**os.environ, "OLLAMA_HOST": "127.0.0.1:11434", "OLLAMA_KEEP_ALIVE": "-1"},
)
for _ in range(60):
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=2)
        print("ollama ready"); break
    except Exception:
        time.sleep(1)
else:
    print(open("/tmp/ollama.log").read())


In [ ]:
#@title 3 · Pull the models (several minutes the first time)
!ollama pull {CODE_MODEL}
!ollama pull {VISION_MODEL}
!ollama list


In [ ]:
#@title 4 · Fetch the router
!rm -rf /content/cloud-ai && git clone --depth 1 {REPO} /content/cloud-ai
!pip -q install -r /content/cloud-ai/backend/requirements.txt


In [ ]:
#@title 5 · Start the Cloud AI router
server_proc = subprocess.Popen(
    ["python", "/content/cloud-ai/backend/server.py"],
    stdout=open("/tmp/cloud.log", "w"), stderr=subprocess.STDOUT,
    env={**os.environ, "CLOUD_HOST": "127.0.0.1", "CLOUD_PORT": "7655"},
)
for _ in range(30):
    try:
        req = urllib.request.Request("http://127.0.0.1:7655/health",
                                     headers={"Authorization": "Bearer " + os.environ["CLOUD_TOKEN"]})
        print(urllib.request.urlopen(req, timeout=2).read().decode()); break
    except Exception:
        time.sleep(1)
else:
    print(open("/tmp/cloud.log").read())


In [ ]:
#@title 6 · Open the Cloudflare tunnel and print your VS Code settings
!curl -fsSL -o /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /usr/local/bin/cloudflared
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:7655", "--no-autoupdate"],
    stdout=open("/tmp/cloudflared.log", "w"), stderr=subprocess.STDOUT,
)
url = None
for _ in range(60):
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/tmp/cloudflared.log").read())
    if m:
        url = m.group(0); break
    time.sleep(1)
os.environ["CLOUD_URL"] = url or ""
print("=" * 64)
print("CLOUD_URL=" + str(url))
print("CLOUD_TOKEN=" + os.environ["CLOUD_TOKEN"])
print("=" * 64)
print("VS Code -> Settings -> Cloud AI -> paste Tunnel Url and Token. Keep this notebook running.")


In [ ]:
#@title 7 · Verify through the tunnel
!curl -s -H "Authorization: Bearer $CLOUD_TOKEN" "$CLOUD_URL/health" | python -m json.tool
